### **Bronze Layer**

In [0]:
path = "/Volumes/workspace/brightwatt/raw/readings/"

df = (spark.read.option("header",True).option("inferSchema",True).csv(path))

df.printSchema()                      # what columns and types Spark found
df.show(5, truncate=False)            # peek at the first 5 rows
print(df.count(), "rows in this one file")


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

In [0]:
from pyspark.sql.functions import current_timestamp, col

readings = (spark.read
              .option("header", True)          # keep the column names; NO inferSchema → all stays raw text
              .csv("/Volumes/workspace/brightwatt/raw/readings/")   # the FOLDER = all 32 files at once
              .withColumn("_ingested_at", current_timestamp())      # when we loaded it
              .withColumn("_source_file", col("_metadata.file_path"))    
              )

readings.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.readings")

print(spark.table("workspace.bronze.readings").count(), "rows")
spark.table("workspace.bronze.readings").printSchema()

In [0]:
invoices = (spark.read
              .option("multiLine", True)   # invoices.json is a formatted array, not one-object-per-line
              .json("/Volumes/workspace/brightwatt/raw/invoices.json")
              .withColumn("_ingested_at", current_timestamp()))

invoices.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.invoices")
print(spark.table("workspace.bronze.invoices").count(), "invoices")   # 50

In [0]:
drift = (spark.read
           .option("header", True)
           .csv("/Volumes/workspace/brightwatt/raw/readings_drift/")
           .withColumn("_ingested_at", current_timestamp()))

drift.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.readings_drift")
print(drift.columns)   # ['reading_id','meter_id','ts','kwh','read_type','source_system', ...]

In [0]:
def land_csv(name):
    (spark.read.option("header", True)
    .csv(f"/Volumes/workspace/brightwatt/raw/{name}.csv")
    .withColumn("_ingested_at", current_timestamp())
    .write.format("delta").mode("overwrite")
    .saveAsTable(f"workspace.bronze.{name}"))
    print(f"bronze.{name}:", spark.table(f"workspace.bronze.{name}").count(), "rows")

for name in ["customers", "meters", "tariffs", "payments"]:
    land_csv(name)


In [0]:
display(spark.sql("SHOW TABLES IN workspace.bronze"))

main  = spark.table("workspace.bronze.readings").count()
drift = spark.table("workspace.bronze.readings_drift").count()
print("readings + drift =", main + drift)   # 84,577